In [1]:
"""torch.compile(fullgraph=True) can't proxy a tensor subclass without __tensor_flatten__."""

import torch


class ScaledTensor(torch.Tensor):
    @staticmethod
    def __new__(cls, data, scale):
        t = torch.Tensor._make_wrapper_subclass(cls, data.shape, dtype=data.dtype, device=data.device)
        t._data = data
        t._scale = scale
        return t

    def __init__(self, data, scale):
        pass

    @classmethod
    def __torch_dispatch__(cls, func, types, args=(), kwargs=None):
        def unwrap(t):
            return t._data if isinstance(t, ScaledTensor) else t
        return func(*torch.utils._pytree.tree_map(unwrap, args),
                    **torch.utils._pytree.tree_map(unwrap, kwargs or {}))


@torch.library.custom_op("mylib::double", mutates_args=[])
def double(x: torch.Tensor) -> torch.Tensor:
    return x * 2

@double.register_fake
def _(x):
    return torch.empty_like(x)


def fn(x):
    return torch.ops.mylib.double(x)


t = ScaledTensor(torch.randn(4, device="cuda"), scale=0.5)

print("eager:", fn(t))
print("compile:", torch.compile(fn, fullgraph=True)(t))


eager: tensor([-0.8412,  0.8878,  2.7149, -1.2563], device='cuda:0')


Unsupported: Failed to convert args/kwargs to proxy
  Explanation: Missing `as_proxy()` implementation for some arg/kwarg.


  Developer debug context: call_function args: LazyVariableTracker(realized: UserDefinedObjectVariable(ScaledTensor)) 

 For more details about this graph break, please visit: https://meta-pytorch.github.io/compile-graph-break-site/gb/gb0055.html

from user code:
   File "/tmp/ipykernel_544398/3283391580.py", line 35, in fn
    return torch.ops.mylib.double(x)

Set TORCHDYNAMO_VERBOSE=1 for the internal stack trace (please do this especially if you're reporting a bug to PyTorch). For even more developer context, set TORCH_LOGS="+dynamo"
